# Simulation functions

## Module

In [3]:
import random
import math
import numpy as np

## Symbols

In [5]:
# N: population size
# n: number of preferences
# r_1: number of phenotype on layer-1
# r_2: number of phenotype on layer-2
# beta: selection strength
# u: strategy(preference) mutation rate
# v: phenotypic mutation rate
# R_1, S_1, T_1, P_1: payoff matrix of layer-1
# R_2, S_2, T_2, P_2: payoff matrix of layer-2

## Functions

### Non-concurrent mutation and independence

In [8]:
def Simulation_Non_concurrent_Independence(N, n, r_1, r_2, beta, u, v, R_1, S_1, T_1, P_1, R_2, S_2, T_2, P_2):
    
    class Individual:
        def __init__(self):
            self.preference_p = random.randint(1, n) / n
            self.preference_q = random.randint(1, n) / n
            self.phenotype_1 = random.randint(1, r_1)
            self.phenotype_2 = random.randint(1, r_2)
            self.payoff = 0
            self.fitness = 0
    
    # initialize population
    def initialize_population(N):
        return [Individual() for _ in range(N)]
    
    # update preference count
    def update_preference_count(population, preference_p_count, preference_q_count):
        for individual in population:
            index_p = int(individual.preference_p * n) - 1
            index_q = int(individual.preference_q * n) - 1
            preference_p_count[index_p] += 1
            preference_q_count[index_q] += 1
    
    # calculate payoff for each individual
    def calculate_payoff(population):
        for i in range(N):
            individual_i = population[i]
            p_i = individual_i.preference_p
            q_i = individual_i.preference_q
            individual_i.payoff = 0
        for i in range(N):
            for j in range(i+1, N):
                individual_j = population[j]
                p_j = individual_j.preference_p
                q_j = individual_j.preference_q
                if individual_i.phenotype_1 == individual_j.phenotype_1:
                    individual_i.payoff += R_1*p_i*p_j + S_1*p_i*(1-p_j) + T_1*(1-p_i)*p_j + P_1*(1-p_i)*(1-p_j)
                    individual_j.payoff += R_1*p_j*p_i + S_1*p_j*(1-p_i) + T_1*(1-p_j)*p_i + P_1*(1-p_j)*(1-p_i)
                if individual_i.phenotype_2 == individual_j.phenotype_2:
                    individual_i.payoff += R_2*q_i*q_j + S_2*q_i*(1-q_j) + T_2*(1-q_i)*q_j + P_2*(1-q_i)*(1-q_j)
                    individual_j.payoff += R_2*q_j*q_i + S_2*q_j*(1-q_i) + T_2*(1-q_j)*q_i + P_2*(1-q_j)*(1-q_i)
    
    # calculate fitness for each individual
    def calculate_fitness(population):
        for individual in population:
            individual.fitness = math.exp(beta * individual.payoff)
    
    # normalize fitness values
    def normalize_fitness(population):
        total_fitness = sum(individual.fitness for individual in population)
        for individual in population:
            individual.fitness /= total_fitness
    
    # select parent based on fitness
    def select_parent(population):
        rand = random.uniform(0, 1)
        cumulative_prob = 0
        for individual in population:
            cumulative_prob += individual.fitness
            if cumulative_prob >= rand:
                return individual
        return population[-1]
    
    # evolve the offspring
    def evolve_offspring(parent, population):
        random_individual = random.choice(population)
        
        if random.uniform(0, 1) < u:
            random_individual.preference_p = random.randint(1, n) / n
            random_individual.preference_q = random.randint(1, n) / n
        else:
            random_individual.preference_p = parent.preference_p
            random_individual.preference_q = parent.preference_q
        
        if random.uniform(0, 1) < v:
            random_individual.phenotype_1 = random.randint(1, r_1)
        else:
            random_individual.phenotype_1 = parent.phenotype_1
        
        if random.uniform(0, 1) < v:
            random_individual.phenotype_2 = random.randint(1, r_2)
        else:
            random_individual.phenotype_2 = parent.phenotype_2
    
    # evolve one generation (this was the part inside the while loop)
    def evolve_generation(population, preference_p_count, preference_q_count):
        calculate_payoff(population)
        calculate_fitness(population)
        normalize_fitness(population)
        
        parent = select_parent(population)
        evolve_offspring(parent, population)
        
        update_preference_count(population, preference_p_count, preference_q_count)
    
    # main loop
    def run_evolution():
        population = initialize_population(N)
        preference_p_count = [0] * n
        preference_q_count = [0] * n
        update_preference_count(population, preference_p_count, preference_q_count)
        
        update_round = 0
        # ROUND = 500000000
        while update_round < ROUND:
            evolve_generation(population, preference_p_count, preference_q_count)
            update_round += 1
        
        preference_p_ratio = [num / (update_round * N) for num in preference_p_count]
        preference_q_ratio = [num / (update_round * N) for num in preference_q_count]
        return preference_p_ratio, preference_q_ratio
    
    # run the simulation
    preference_p_ratio, preference_q_ratio = run_evolution()
    print(preference_p_ratio, preference_q_ratio)
    return preference_p_ratio, preference_q_ratio

def calculate_expectation(preference_p_ratio, preference_q_ratio):
    n_p = len(preference_p_ratio)
    n_q = len(preference_q_ratio)
    expectation_p = 0
    expectation_q = 0
    for i in range(n_p):
        expectation_p += preference_p_ratio[i] * (i + 1) / n_p
    for i in range(n_q):
        expectation_q += preference_q_ratio[i] * (i + 1) / n_q
    print(expectation_p, expectation_q)
    return expectation_p, expectation_q
    

### Non-concurrent mutation and unidirectional influence ($K=0$)

In [10]:
def Simulation_Non_concurrent_Unidirectional_0(N, n, r_1, r_2, beta, u, v, R_1, S_1, T_1, P_1, R_2, S_2, T_2, P_2):
    # K = 0
    class Individual:
        def __init__(self):
            self.preference_p = random.randint(1, n) / n
            self.preference_q = random.randint(1, n) / n
            self.phenotype_1 = random.randint(1, r_1)
            self.phenotype_2 = random.randint(1, r_2)
            self.payoff = 0
            self.fitness = 0
    
    # initialize population
    def initialize_population(N):
        return [Individual() for _ in range(N)]
    
    # update preference count
    def update_preference_count(population, preference_p_count, preference_q_count):
        for individual in population:
            index_p = int(individual.preference_p * n) - 1
            index_q = int(individual.preference_q * n) - 1
            preference_p_count[index_p] += 1
            preference_q_count[index_q] += 1
    
    # calculate payoff for each individual
    def calculate_payoff(population):
        for i in range(N):
            individual_i = population[i]
            p_i = individual_i.preference_p
            q_i = individual_i.preference_q
            individual_i.payoff = 0
        for i in range(N):
            for j in range(i+1, N):
                individual_j = population[j]
                p_j = individual_j.preference_p
                q_j = individual_j.preference_q
                if individual_i.phenotype_1 == individual_j.phenotype_1:
                    individual_i.payoff += R_1*p_i*p_j + S_1*p_i*(1-p_j) + T_1*(1-p_i)*p_j + P_1*(1-p_i)*(1-p_j)
                    individual_j.payoff += R_1*p_j*p_i + S_1*p_j*(1-p_i) + T_1*(1-p_j)*p_i + P_1*(1-p_j)*(1-p_i)
                if individual_i.phenotype_2 == individual_j.phenotype_2:
                    individual_i.payoff += R_2*q_i*q_j + S_2*q_i*(1-q_j) + T_2*(1-q_i)*q_j + P_2*(1-q_i)*(1-q_j)
                    individual_j.payoff += R_2*q_j*q_i + S_2*q_j*(1-q_i) + T_2*(1-q_j)*q_i + P_2*(1-q_j)*(1-q_i)
    
    # calculate fitness for each individual
    def calculate_fitness(population):
        for individual in population:
            individual.fitness = math.exp(beta * individual.payoff)
    
    # normalize fitness values
    def normalize_fitness(population):
        total_fitness = sum(individual.fitness for individual in population)
        for individual in population:
            individual.fitness /= total_fitness
    
    # select parent based on fitness
    def select_parent(population):
        rand = random.uniform(0, 1)
        cumulative_prob = 0
        for individual in population:
            cumulative_prob += individual.fitness
            if cumulative_prob >= rand:
                return individual
        return population[-1]
    
    # evolve the offspring
    def evolve_offspring(parent, population):
        random_individual = random.choice(population)
        
        if random.uniform(0, 1) < u:
            random_individual.preference_p = random.randint(1, n) / n
            random_individual.preference_q = random.randint(1, n) / n
        else:
            random_individual.preference_p = parent.preference_p
            random_individual.preference_q = parent.preference_q

        random_1 = random.uniform(0, 1)
        random_2 = random.uniform(0, 1)
        if random_1 < v and random_2 < v:
            random_individual.phenotype_1 = random.randint(1, r_1)
            random_individual.phenotype_2 = random_individual.phenotype_1
        elif random_1 < v and random_2 >= v:
            random_individual.phenotype_1 = random.randint(1, r_1)
            random_individual.phenotype_2 = parent.phenotype_2
        elif random_1 >= v and random_2 < v:
            random_individual.phenotype_1 = parent.phenotype_1
            random_individual.phenotype_2 = random_individual.phenotype_1
        else:
            random_individual.phenotype_1 = parent.phenotype_1
            random_individual.phenotype_2 = parent.phenotype_2
    
    # evolve one generation (this was the part inside the while loop)
    def evolve_generation(population, preference_p_count, preference_q_count):
        calculate_payoff(population)
        calculate_fitness(population)
        normalize_fitness(population)
        
        parent = select_parent(population)
        evolve_offspring(parent, population)
        
        update_preference_count(population, preference_p_count, preference_q_count)
    
    # main loop
    def run_evolution():
        population = initialize_population(N)
        preference_p_count = [0] * n
        preference_q_count = [0] * n
        update_preference_count(population, preference_p_count, preference_q_count)
        
        update_round = 0
        # ROUND = 500000000
        while update_round < ROUND:
            evolve_generation(population, preference_p_count, preference_q_count)
            update_round += 1
        
        preference_p_ratio = [num / (update_round * N) for num in preference_p_count]
        preference_q_ratio = [num / (update_round * N) for num in preference_q_count]
        return preference_p_ratio, preference_q_ratio
    
    # run the simulation
    preference_p_ratio, preference_q_ratio = run_evolution()
    print(preference_p_ratio, preference_q_ratio)
    return preference_p_ratio, preference_q_ratio

def calculate_expectation(preference_p_ratio, preference_q_ratio):
    n_p = len(preference_p_ratio)
    n_q = len(preference_q_ratio)
    expectation_p = 0
    expectation_q = 0
    for i in range(n_p):
        expectation_p += preference_p_ratio[i] * (i + 1) / n_p
    for i in range(n_q):
        expectation_q += preference_q_ratio[i] * (i + 1) / n_q
    print(expectation_p, expectation_q)
    return expectation_p, expectation_q


### Non-concurrent mutation and unidirectional influence ($K=1$)

In [12]:
def Simulation_Non_concurrent_Unidirectional_1(N, n, r_1, r_2, beta, u, v, R_1, S_1, T_1, P_1, R_2, S_2, T_2, P_2):
    # K = 1
    class Individual:
        def __init__(self):
            self.preference_p = random.randint(1, n) / n
            self.preference_q = random.randint(1, n) / n
            self.phenotype_1 = random.randint(1, r_1)
            self.phenotype_2 = random.randint(0, r_1+1)
            self.payoff = 0
            self.fitness = 0
    
    # initialize population
    def initialize_population(N):
        return [Individual() for _ in range(N)]
    
    # update preference count
    def update_preference_count(population, preference_p_count, preference_q_count):
        for individual in population:
            index_p = int(individual.preference_p * n) - 1
            index_q = int(individual.preference_q * n) - 1
            preference_p_count[index_p] += 1
            preference_q_count[index_q] += 1
    
    # calculate payoff for each individual
    def calculate_payoff(population):
        for i in range(N):
            individual_i = population[i]
            p_i = individual_i.preference_p
            q_i = individual_i.preference_q
            individual_i.payoff = 0
        for i in range(N):
            for j in range(i+1, N):
                individual_j = population[j]
                p_j = individual_j.preference_p
                q_j = individual_j.preference_q
                if individual_i.phenotype_1 == individual_j.phenotype_1:
                    individual_i.payoff += R_1*p_i*p_j + S_1*p_i*(1-p_j) + T_1*(1-p_i)*p_j + P_1*(1-p_i)*(1-p_j)
                    individual_j.payoff += R_1*p_j*p_i + S_1*p_j*(1-p_i) + T_1*(1-p_j)*p_i + P_1*(1-p_j)*(1-p_i)
                if individual_i.phenotype_2 == individual_j.phenotype_2:
                    individual_i.payoff += R_2*q_i*q_j + S_2*q_i*(1-q_j) + T_2*(1-q_i)*q_j + P_2*(1-q_i)*(1-q_j)
                    individual_j.payoff += R_2*q_j*q_i + S_2*q_j*(1-q_i) + T_2*(1-q_j)*q_i + P_2*(1-q_j)*(1-q_i)
    
    # calculate fitness for each individual
    def calculate_fitness(population):
        for individual in population:
            individual.fitness = math.exp(beta * individual.payoff)
    
    # normalize fitness values
    def normalize_fitness(population):
        total_fitness = sum(individual.fitness for individual in population)
        for individual in population:
            individual.fitness /= total_fitness
    
    # select parent based on fitness
    def select_parent(population):
        rand = random.uniform(0, 1)
        cumulative_prob = 0
        for individual in population:
            cumulative_prob += individual.fitness
            if cumulative_prob >= rand:
                return individual
        return population[-1]
    
    # evolve the offspring
    def evolve_offspring(parent, population):
        random_individual = random.choice(population)
        
        if random.uniform(0, 1) < u:
            random_individual.preference_p = random.randint(1, n) / n
            random_individual.preference_q = random.randint(1, n) / n
        else:
            random_individual.preference_p = parent.preference_p
            random_individual.preference_q = parent.preference_q

        random_1 = random.uniform(0, 1)
        random_2 = random.uniform(0, 1)
        if random_1 < v and random_2 < v:
            random_individual.phenotype_1 = random.randint(1, r_1)
            if random_individual.phenotype_1 == 1:
                random_individual.phenotype_2 = random.choice([0, 1, 2])
            elif random_individual.phenotype_1 == r_1:
                random_individual.phenotype_2 = random.choice([r_1-1, r_1, r_1+1])
            else:
                random_individual.phenotype_2 = random.randint(random_individual.phenotype_1-1, random_individual.phenotype_1+1)
        elif random_1 < v and random_2 >= v:
            random_individual.phenotype_1 = random.randint(1, r_1)
            random_individual.phenotype_2 = parent.phenotype_2
        elif random_1 >= v and random_2 < v:
            random_individual.phenotype_1 = parent.phenotype_1
            if random_individual.phenotype_1 == 1:
                random_individual.phenotype_2 = random.choice([0, 1, 2])
            elif random_individual.phenotype_1 == r_1:
                random_individual.phenotype_2 = random.choice([r_1-1, r_1, r_1+1])
            else:
                random_individual.phenotype_2 = random.randint(random_individual.phenotype_1-1, random_individual.phenotype_1+1)
        else:
            random_individual.phenotype_1 = parent.phenotype_1
            random_individual.phenotype_2 = parent.phenotype_2
    
    # evolve one generation (this was the part inside the while loop)
    def evolve_generation(population, preference_p_count, preference_q_count):
        calculate_payoff(population)
        calculate_fitness(population)
        normalize_fitness(population)
        
        parent = select_parent(population)
        evolve_offspring(parent, population)
        
        update_preference_count(population, preference_p_count, preference_q_count)
    
    # main loop
    def run_evolution():
        population = initialize_population(N)
        preference_p_count = [0] * n
        preference_q_count = [0] * n
        update_preference_count(population, preference_p_count, preference_q_count)
        
        update_round = 0
        # ROUND = 500000000
        while update_round < ROUND:
            evolve_generation(population, preference_p_count, preference_q_count)
            update_round += 1
        
        preference_p_ratio = [num / (update_round * N) for num in preference_p_count]
        preference_q_ratio = [num / (update_round * N) for num in preference_q_count]
        return preference_p_ratio, preference_q_ratio
    
    # run the simulation
    preference_p_ratio, preference_q_ratio = run_evolution()
    print(preference_p_ratio, preference_q_ratio)
    return preference_p_ratio, preference_q_ratio

def calculate_expectation(preference_p_ratio, preference_q_ratio):
    n_p = len(preference_p_ratio)
    n_q = len(preference_q_ratio)
    expectation_p = 0
    expectation_q = 0
    for i in range(n_p):
        expectation_p += preference_p_ratio[i] * (i + 1) / n_p
    for i in range(n_q):
        expectation_q += preference_q_ratio[i] * (i + 1) / n_q
    print(expectation_p, expectation_q)
    return expectation_p, expectation_q


### Non-concurrent mutation and bidirectional influence ($K=1$)

In [14]:
def Simulation_Non_concurrent_Bidirectional(N, n, r_1, r_2, beta, u, v, R_1, S_1, T_1, P_1, R_2, S_2, T_2, P_2):
    # K = 1
    class Individual:
        def __init__(self):
            self.preference_p = random.randint(1, n) / n
            self.preference_q = random.randint(1, n) / n
            self.phenotype_1 = int(np.round(np.random.normal(loc=0, scale=1)))
            self.phenotype_2 = int(np.round(np.random.normal(loc=0, scale=1)))
            self.payoff = 0
            self.fitness = 0
    
    # initialize population
    def initialize_population(N):
        return [Individual() for _ in range(N)]
    
    # update preference count
    def update_preference_count(population, preference_p_count, preference_q_count):
        for individual in population:
            index_p = int(individual.preference_p * n) - 1
            index_q = int(individual.preference_q * n) - 1
            preference_p_count[index_p] += 1
            preference_q_count[index_q] += 1
    
    # calculate payoff for each individual
    def calculate_payoff(population):
        for i in range(N):
            individual_i = population[i]
            p_i = individual_i.preference_p
            q_i = individual_i.preference_q
            individual_i.payoff = 0
        for i in range(N):
            for j in range(i+1, N):
                individual_j = population[j]
                p_j = individual_j.preference_p
                q_j = individual_j.preference_q
                if individual_i.phenotype_1 == individual_j.phenotype_1:
                    individual_i.payoff += R_1*p_i*p_j + S_1*p_i*(1-p_j) + T_1*(1-p_i)*p_j + P_1*(1-p_i)*(1-p_j)
                    individual_j.payoff += R_1*p_j*p_i + S_1*p_j*(1-p_i) + T_1*(1-p_j)*p_i + P_1*(1-p_j)*(1-p_i)
                if individual_i.phenotype_2 == individual_j.phenotype_2:
                    individual_i.payoff += R_2*q_i*q_j + S_2*q_i*(1-q_j) + T_2*(1-q_i)*q_j + P_2*(1-q_i)*(1-q_j)
                    individual_j.payoff += R_2*q_j*q_i + S_2*q_j*(1-q_i) + T_2*(1-q_j)*q_i + P_2*(1-q_j)*(1-q_i)
    
    # calculate fitness for each individual
    def calculate_fitness(population):
        for individual in population:
            individual.fitness = math.exp(beta * individual.payoff)
    
    # normalize fitness values
    def normalize_fitness(population):
        total_fitness = sum(individual.fitness for individual in population)
        for individual in population:
            individual.fitness /= total_fitness
    
    # select parent based on fitness
    def select_parent(population):
        rand = random.uniform(0, 1)
        cumulative_prob = 0
        for individual in population:
            cumulative_prob += individual.fitness
            if cumulative_prob >= rand:
                return individual
        return population[-1]
    
    # evolve the offspring
    def evolve_offspring(parent, population):
        random_individual = random.choice(population)
        
        if random.uniform(0, 1) < u:
            random_individual.preference_p = random.randint(1, n) / n
            random_individual.preference_q = random.randint(1, n) / n
        else:
            random_individual.preference_p = parent.preference_p
            random_individual.preference_q = parent.preference_q

        random_1 = random.uniform(0, 1)
        random_2 = random.uniform(0, 1)
        if random_1 < v and random_2 < v:
            random_3 = random.uniform(0, 1)
            if random_3 < 0.5:
                random_individual.phenotype_1 = random.randint(random_individual.phenotype_2-1, random_individual.phenotype_2+1)
                random_individual.phenotype_2 = random.randint(random_individual.phenotype_1-1, random_individual.phenotype_1+1)
            else:
                random_individual.phenotype_2 = random.randint(random_individual.phenotype_1-1, random_individual.phenotype_1+1)
                random_individual.phenotype_1 = random.randint(random_individual.phenotype_2-1, random_individual.phenotype_2+1)
        elif random_1 < v and random_2 >= v:
            random_individual.phenotype_2 = parent.phenotype_2
            random_individual.phenotype_1 = random.randint(random_individual.phenotype_2-1, random_individual.phenotype_2+1)
        elif random_1 >= v and random_2 < v:
            random_individual.phenotype_1 = parent.phenotype_1
            random_individual.phenotype_2 = random.randint(random_individual.phenotype_1-1, random_individual.phenotype_1+1)
        else:
            random_individual.phenotype_1 = parent.phenotype_1
            random_individual.phenotype_2 = parent.phenotype_2
    
    # evolve one generation (this was the part inside the while loop)
    def evolve_generation(population, preference_p_count, preference_q_count):
        calculate_payoff(population)
        calculate_fitness(population)
        normalize_fitness(population)
        
        parent = select_parent(population)
        evolve_offspring(parent, population)
        
        update_preference_count(population, preference_p_count, preference_q_count)
    
    # main loop
    def run_evolution():
        population = initialize_population(N)
        preference_p_count = [0] * n
        preference_q_count = [0] * n
        update_preference_count(population, preference_p_count, preference_q_count)
        
        update_round = 0
        # ROUND = 1000000000
        while update_round < ROUND:
            evolve_generation(population, preference_p_count, preference_q_count)
            update_round += 1
        
        preference_p_ratio = [num / (update_round * N) for num in preference_p_count]
        preference_q_ratio = [num / (update_round * N) for num in preference_q_count]
        return preference_p_ratio, preference_q_ratio
    
    # run the simulation
    preference_p_ratio, preference_q_ratio = run_evolution()
    print(preference_p_ratio, preference_q_ratio)
    return preference_p_ratio, preference_q_ratio

def calculate_expectation(preference_p_ratio, preference_q_ratio):
    n_p = len(preference_p_ratio)
    n_q = len(preference_q_ratio)
    expectation_p = 0
    expectation_q = 0
    for i in range(n_p):
        expectation_p += preference_p_ratio[i] * (i + 1) / n_p
    for i in range(n_q):
        expectation_q += preference_q_ratio[i] * (i + 1) / n_q
    print(expectation_p, expectation_q)
    return expectation_p, expectation_q


### Concurrent mutation and independence

In [16]:
def Simulation_Concurrent_Independence(N, n, r_1, r_2, beta, u, v, R_1, S_1, T_1, P_1, R_2, S_2, T_2, P_2):
    
    class Individual:
        def __init__(self):
            self.preference_p = random.randint(1, n) / n
            self.preference_q = random.randint(1, n) / n
            self.phenotype_1 = random.randint(1, r_1)
            self.phenotype_2 = random.randint(1, r_2)
            self.payoff = 0
            self.fitness = 0
    
    # initialize population
    def initialize_population(N):
        return [Individual() for _ in range(N)]
    
    # update preference count
    def update_preference_count(population, preference_p_count, preference_q_count):
        for individual in population:
            index_p = int(individual.preference_p * n) - 1
            index_q = int(individual.preference_q * n) - 1
            preference_p_count[index_p] += 1
            preference_q_count[index_q] += 1
    
    # calculate payoff for each individual
    def calculate_payoff(population):
        for i in range(N):
            individual_i = population[i]
            p_i = individual_i.preference_p
            q_i = individual_i.preference_q
            individual_i.payoff = 0
        for i in range(N):
            for j in range(i+1, N):
                individual_j = population[j]
                p_j = individual_j.preference_p
                q_j = individual_j.preference_q
                if individual_i.phenotype_1 == individual_j.phenotype_1:
                    individual_i.payoff += R_1*p_i*p_j + S_1*p_i*(1-p_j) + T_1*(1-p_i)*p_j + P_1*(1-p_i)*(1-p_j)
                    individual_j.payoff += R_1*p_j*p_i + S_1*p_j*(1-p_i) + T_1*(1-p_j)*p_i + P_1*(1-p_j)*(1-p_i)
                if individual_i.phenotype_2 == individual_j.phenotype_2:
                    individual_i.payoff += R_2*q_i*q_j + S_2*q_i*(1-q_j) + T_2*(1-q_i)*q_j + P_2*(1-q_i)*(1-q_j)
                    individual_j.payoff += R_2*q_j*q_i + S_2*q_j*(1-q_i) + T_2*(1-q_j)*q_i + P_2*(1-q_j)*(1-q_i)
    
    # calculate fitness for each individual
    def calculate_fitness(population):
        for individual in population:
            individual.fitness = math.exp(beta * individual.payoff)
    
    # normalize fitness values
    def normalize_fitness(population):
        total_fitness = sum(individual.fitness for individual in population)
        for individual in population:
            individual.fitness /= total_fitness
    
    # select parent based on fitness
    def select_parent(population):
        rand = random.uniform(0, 1)
        cumulative_prob = 0
        for individual in population:
            cumulative_prob += individual.fitness
            if cumulative_prob >= rand:
                return individual
        return population[-1]
    
    # evolve the offspring
    def evolve_offspring(parent, population):
        random_individual = random.choice(population)
        
        if random.uniform(0, 1) < u:
            random_individual.preference_p = random.randint(1, n) / n
            random_individual.preference_q = random.randint(1, n) / n
        else:
            random_individual.preference_p = parent.preference_p
            random_individual.preference_q = parent.preference_q
        
        if random.uniform(0, 1) < v:
            random_individual.phenotype_1 = random.randint(1, r_1)
            random_individual.phenotype_2 = random.randint(1, r_2)
        else:
            random_individual.phenotype_1 = parent.phenotype_1
            random_individual.phenotype_2 = parent.phenotype_2
    
    # evolve one generation (this was the part inside the while loop)
    def evolve_generation(population, preference_p_count, preference_q_count):
        calculate_payoff(population)
        calculate_fitness(population)
        normalize_fitness(population)
        
        parent = select_parent(population)
        evolve_offspring(parent, population)
        
        update_preference_count(population, preference_p_count, preference_q_count)
    
    # main loop
    def run_evolution():
        population = initialize_population(N)
        preference_p_count = [0] * n
        preference_q_count = [0] * n
        update_preference_count(population, preference_p_count, preference_q_count)
        
        update_round = 0
        # ROUND = 500000000
        while update_round < ROUND:
            evolve_generation(population, preference_p_count, preference_q_count)
            update_round += 1
        
        preference_p_ratio = [num / (update_round * N) for num in preference_p_count]
        preference_q_ratio = [num / (update_round * N) for num in preference_q_count]
        return preference_p_ratio, preference_q_ratio
    
    # run the simulation
    preference_p_ratio, preference_q_ratio = run_evolution()
    print(preference_p_ratio, preference_q_ratio)
    return preference_p_ratio, preference_q_ratio

def calculate_expectation(preference_p_ratio, preference_q_ratio):
    n_p = len(preference_p_ratio)
    n_q = len(preference_q_ratio)
    expectation_p = 0
    expectation_q = 0
    for i in range(n_p):
        expectation_p += preference_p_ratio[i] * (i + 1) / n_p
    for i in range(n_q):
        expectation_q += preference_q_ratio[i] * (i + 1) / n_q
    print(expectation_p, expectation_q)
    return expectation_p, expectation_q
    